# PageRank Nibble / Approximate Personalized PageRank

In [ ]:
%pip install networkx pandas

import heapq
import json
from collections import defaultdict
from pathlib import Path

import networkx as nx
import pandas as pd

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "subreddit_similarity_results.csv").exists():
    ROOT = ROOT.parent

RESULT_DIR = ROOT / "result"
SIM_CSV = ROOT / "subreddit_similarity_results.csv"
COMMUNITY_CSV = RESULT_DIR / "community_result.csv"
CLUSTER_NAMES_CSV = RESULT_DIR / "cluster_names.csv"
BRIDGE_CSV = RESULT_DIR / "bridge_result.csv"
GATEWAY_CSV = RESULT_DIR / "gateway_result.csv"
HIGHWAY_CSV = RESULT_DIR / "highway_result.csv"

ALPHA = 0.15
EPS = 1e-4
SEED_COUNT = 12
MIN_CLUSTER_SIZE = 2
MAX_CLUSTER_SIZE = 80

In [ ]:
df_sim = pd.read_csv(SIM_CSV)
threshold = df_sim["Similarity_Score"].quantile(0.97)
df_edges = df_sim[df_sim["Similarity_Score"] >= threshold]

G = nx.from_pandas_edgelist(
    df_edges,
    source="Subreddit_A",
    target="Subreddit_B",
    edge_attr="Similarity_Score",
    create_using=nx.Graph(),
)
for _, _, data in G.edges(data=True):
    data["weight"] = float(data["Similarity_Score"])

degree = dict(G.degree(weight="weight"))
total_volume = sum(degree.values())
print(f"Threshold P97: {threshold:.4f}")
print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

In [ ]:
def top_subreddits(path, score_col):
    if not path.exists():
        return []
    df = pd.read_csv(path).sort_values(score_col, ascending=False)
    return df["subreddit"].dropna().astype(str).tolist()


def highway_subreddits(path):
    if not path.exists():
        return []
    nodes = []
    for text in pd.read_csv(path).sort_values("rank")["highway_nodes"].fillna(""):
        text = str(text).replace("\u2192", "->").replace("\u00e2\u2020\u2019", "->")
        nodes.extend(part.strip() for part in text.split("->") if part.strip())
    return nodes


candidates = (
    top_subreddits(BRIDGE_CSV, "bridge_score")
    + top_subreddits(GATEWAY_CSV, "gateway_score_normalized")
    + highway_subreddits(HIGHWAY_CSV)
    + [node for node, _ in sorted(degree.items(), key=lambda x: x[1], reverse=True)]
)

seeds = []
for node in candidates:
    if node in G and node not in seeds:
        seeds.append(node)
    if len(seeds) == SEED_COUNT:
        break

print(seeds)

In [ ]:
def approximate_ppr(seed, alpha=ALPHA, eps=EPS):
    p, r = defaultdict(float), defaultdict(float)
    r[seed] = 1.0
    heap = [(-r[seed] / degree[seed], seed)]
    pushes = 0

    def priority(node):
        return r[node] / degree[node] if degree[node] else 0.0

    while heap:
        _, node = heapq.heappop(heap)
        if priority(node) <= eps:
            continue

        residual = r[node]
        p[node] += alpha * residual
        remain = (1 - alpha) * residual
        r[node] = remain / 2
        pushes += 1

        for nbr, data in G[node].items():
            old = priority(nbr)
            r[nbr] += remain * data["weight"] / (2 * degree[node])
            if old <= eps < priority(nbr):
                heapq.heappush(heap, (-priority(nbr), nbr))

        if priority(node) > eps:
            heapq.heappush(heap, (-priority(node), node))

    return dict(p), dict(r), pushes


def sweep_cluster(ppr):
    order = sorted(ppr, key=lambda n: ppr[n] / degree[n], reverse=True)[:500]
    S, cut, vol = set(), 0.0, 0.0
    best = ([], 1.0, 0.0, 0.0)

    for i, node in enumerate(order, 1):
        S.add(node)
        vol += degree[node]
        for nbr, data in G[node].items():
            cut += -data["weight"] if nbr in S else data["weight"]

        if MIN_CLUSTER_SIZE <= i <= MAX_CLUSTER_SIZE:
            conductance = cut / min(vol, total_volume - vol)
            if conductance < best[1]:
                best = (order[:i], conductance, cut, vol)

    return best

In [ ]:
community_df = pd.read_csv(COMMUNITY_CSV)
name_df = pd.read_csv(CLUSTER_NAMES_CSV)[["community_id", "name"]]
community_df = community_df.merge(name_df, on="community_id", how="left")
info = community_df.set_index("subreddit").to_dict("index")


def community_value(node, col, default=""):
    return info.get(node, {}).get(col, default)


runs, rows = [], []
for seed in seeds:
    ppr, residual, pushes = approximate_ppr(seed)
    cluster, conductance, cut, vol = sweep_cluster(ppr)
    runs.append({"seed": seed, "cluster": cluster, "conductance": conductance, "cut": cut, "vol": vol, "ppr": ppr, "residual": residual, "pushes": pushes})

    for rank, node in enumerate(cluster, 1):
        rows.append({
            "seed_subreddit": seed,
            "rank": rank,
            "subreddit": node,
            "community_id": community_value(node, "community_id"),
            "community_name": community_value(node, "name"),
            "ppr_score": ppr[node],
            "score_per_degree": ppr[node] / degree[node],
            "weighted_degree": degree[node],
            "cluster_size": len(cluster),
            "cluster_conductance": conductance,
            "cluster_cut": cut,
            "cluster_volume": vol,
            "alpha": ALPHA,
            "epsilon": EPS,
            "min_edge_weight": threshold,
        })

result_df = pd.DataFrame(rows)
summary_df = pd.DataFrame([{
    "seed_subreddit": r["seed"],
    "seed_community_id": community_value(r["seed"], "community_id"),
    "seed_community_name": community_value(r["seed"], "name"),
    "cluster_size": len(r["cluster"]),
    "cluster_conductance": r["conductance"],
    "cluster_cut": r["cut"],
    "cluster_volume": r["vol"],
    "ppr_mass": sum(r["ppr"].values()),
    "residual_mass": sum(r["residual"].values()),
    "push_count": r["pushes"],
    "ranked_node_count": len(r["ppr"]),
    "top_subreddits": " | ".join(r["cluster"][:12]),
    "top_communities": " | ".join(dict.fromkeys(community_value(n, "name") for n in r["cluster"] if community_value(n, "name"))),
    "alpha": ALPHA,
    "epsilon": EPS,
    "min_edge_weight": threshold,
} for r in runs]).sort_values("cluster_conductance")

display(summary_df)

In [ ]:
RESULT_DIR.mkdir(exist_ok=True)
result_df.to_csv(RESULT_DIR / "pagerank_nibble_result.csv", index=False)
summary_df.to_csv(RESULT_DIR / "pagerank_nibble_summary.csv", index=False)

print("Saved:")
print(RESULT_DIR / "pagerank_nibble_result.csv")
print(RESULT_DIR / "pagerank_nibble_summary.csv")

In [ ]:
selected = list(dict.fromkeys(result_df["subreddit"].tolist()))[:220]
selected_set = set(selected)
seed_set = set(seeds)

nodes = [{
    "id": node,
    "label": node,
    "group": "seed" if node in seed_set else "ppr_cluster",
    "shape": "star" if node in seed_set else "dot",
    "size": 28 if node in seed_set else 14,
    "title": f"Subreddit: {node}<br>Community: {community_value(node, 'name')}",
} for node in selected]

edges = []
for u, v, data in G.edges(data=True):
    if u in selected_set and v in selected_set:
        edges.append({"from": u, "to": v, "width": 1, "Similarity_Score": data["weight"]})

payload = {
    "id": "pagerank_nibble",
    "title": "PageRank Nibble local clusters",
    "description": "Approximate Personalized PageRank neighborhoods from selected seeds.",
    "nodes": nodes,
    "edges": edges[:1200],
    "groups": ["seed", "ppr_cluster"],
}

with open(RESULT_DIR / "pagerank_nibble_graph.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(RESULT_DIR / "pagerank_nibble_graph.json")